# Irrigation-v0 — RL Benchmark

Trains **DQN**, **PPO**, and **A2C** from Stable-Baselines3 on a custom Gymnasium environment that simulates multi-zone irrigation scheduling over a 150-day growing season.

**Portfolio artifact:** the `Irrigation-v0` environment itself; algorithms are off-the-shelf SB3.

**Runtime:** ~10 min on Colab CPU with the default timestep budget.

## 1  Setup

In [ ]:
# Clone the repo (skip if already present / running locally)
import os
if not os.path.exists("irrigation-rl"):
    !git clone https://github.com/YOUR_USERNAME/irrigation-rl.git
os.chdir("irrigation-rl")
!pip install -q -e '.[train,plot,dev]'

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import irrigation_env  # registers Irrigation-v0
from irrigation_env.environment import IrrigationEnv
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

from agents.baselines import (
    random_policy, MoistureThresholdPolicy,
    run_episodes, summarize, make_comparison_plot,
)
from agents.train import make_model, eval_model
from agents.evaluate import run_model_episodes

RESULTS = Path("results")
(RESULTS / "models").mkdir(parents=True, exist_ok=True)
(RESULTS / "plots").mkdir(parents=True, exist_ok=True)
(RESULTS / "raw").mkdir(parents=True, exist_ok=True)

print("Environment obs dim:", IrrigationEnv().observation_space.shape)
print("Action space:       ", IrrigationEnv().action_space)

## 2  Baseline policies

Two non-learning reference policies establish the performance floor and a domain-informed ceiling:

| Policy | Description |
|--------|-------------|
| **random** | Uniform over Discrete(256) — lower bound |
| **threshold** | Refills zones below 75% FC toward FC, subtracts forecast rain |

In [ ]:
BASELINE_SEEDS = list(range(20))

baseline_results = []
baseline_results += run_episodes("random",    random_policy,               BASELINE_SEEDS)
baseline_results += run_episodes("threshold", MoistureThresholdPolicy(),   BASELINE_SEEDS)

bl_summary = summarize(baseline_results)
for name, s in bl_summary.items():
    print(f"{name:<12s}  reward {s['reward_mean']:7.2f} ± {s['reward_std']:5.2f}   "
          f"yield {s['yield_mean']:.3f} ± {s['yield_std']:.3f}   "
          f"water {s['water_mean_mm']:6.0f} mm")

## 3  Train RL agents

We train three off-the-shelf SB3 algorithms.  
Tune `TIMESTEPS` up (e.g. 500 000) for better results at the cost of runtime.

In [ ]:
TIMESTEPS = 200_000   # ~10 min on Colab CPU; raise to 500k for publication quality
TRAIN_SEED = 42
ALGOS = ["dqn", "ppo", "a2c"]

In [ ]:
import time, json

trained_models = {}

for algo in ALGOS:
    run_name = f"{algo}_seed{TRAIN_SEED}"
    log_dir  = RESULTS / "logs" / run_name
    log_dir.mkdir(parents=True, exist_ok=True)

    env = make_vec_env(IrrigationEnv, n_envs=1, seed=TRAIN_SEED)
    model = make_model(algo, env, log_dir, TRAIN_SEED)

    print(f"\n{'='*55}")
    print(f"Training {algo.upper():4s}  ({TIMESTEPS:,} steps) …")
    t0 = time.time()
    model.learn(total_timesteps=TIMESTEPS, progress_bar=True)
    print(f"Done in {time.time()-t0:.1f}s")

    model_path = RESULTS / "models" / run_name
    model.save(str(model_path))
    trained_models[algo] = model

    stats = eval_model(model, n_episodes=20)
    print(f"Eval → reward {stats['reward_mean']:.2f} ± {stats['reward_std']:.2f}   "
          f"yield {stats['yield_mean']:.3f}   water {stats['water_mean_mm']:.0f} mm")

    stats.update({"algo": algo, "seed": TRAIN_SEED, "timesteps": TIMESTEPS})
    with open(RESULTS / "raw" / f"train_{run_name}.json", "w") as f:
        json.dump(stats, f, indent=2)

## 4  Evaluation — RL vs baselines

Deterministic rollouts on 30 held-out seeds (2000–2029) never seen during training.

In [ ]:
EVAL_SEEDS = list(range(2000, 2030))

all_results = list(baseline_results)   # start with the baselines we already ran

for algo in ALGOS:
    model_path = RESULTS / "models" / f"{algo}_seed{TRAIN_SEED}"
    all_results += run_model_episodes(model_path, algo, EVAL_SEEDS)

summary = summarize(all_results)
print(f"\n{'Policy':<12s}  {'Reward':>14s}   {'Yield':>14s}   {'Water (mm)':>15s}")
print("-" * 65)
for name, s in summary.items():
    print(f"{name:<12s}  {s['reward_mean']:6.2f} ± {s['reward_std']:5.2f}   "
          f"{s['yield_mean']:.3f} ± {s['yield_std']:.3f}   "
          f"{s['water_mean_mm']:6.0f} ± {s['water_std_mm']:5.0f}")

## 5  Visualisation

In [ ]:
plot_path = RESULTS / "plots" / "eval_comparison.png"
make_comparison_plot(all_results, plot_path)

from IPython.display import Image
Image(str(plot_path))

In [ ]:
from agents.evaluate import make_rl_trajectory_plot

# Show the best RL algorithm vs the threshold heuristic for one season
best_algo = max(ALGOS, key=lambda a: summary.get(a, {}).get("reward_mean", -1e9))
traj_path = RESULTS / "plots" / f"trajectory_{best_algo}.png"
model_path = RESULTS / "models" / f"{best_algo}_seed{TRAIN_SEED}"
make_rl_trajectory_plot(best_algo, model_path, seed=EVAL_SEEDS[0], out_path=traj_path)

Image(str(traj_path))

## 6  Learning curves  (TensorBoard)

Run the cell below to open an inline TensorBoard dashboard showing episodic reward over training.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/logs

## 7  Seed sweep (optional)

Train the best algorithm across multiple seeds to measure variance.  
Skip this cell during a quick demo — it multiplies training time by `N_SEEDS`.

In [ ]:
SWEEP_ALGO  = "ppo"          # algorithm to sweep
SWEEP_SEEDS = [0, 1, 2, 3, 4]
SWEEP_STEPS = 200_000

sweep_results = []
for seed in SWEEP_SEEDS:
    run_name = f"{SWEEP_ALGO}_seed{seed}"
    log_dir  = RESULTS / "logs" / run_name
    log_dir.mkdir(parents=True, exist_ok=True)

    env   = make_vec_env(IrrigationEnv, n_envs=1, seed=seed)
    model = make_model(SWEEP_ALGO, env, log_dir, seed)
    model.learn(total_timesteps=SWEEP_STEPS, progress_bar=False)

    model_path = RESULTS / "models" / run_name
    model.save(str(model_path))

    stats = eval_model(model, n_episodes=20)
    sweep_results.append({"seed": seed, **stats})
    print(f"seed {seed}: reward {stats['reward_mean']:.2f} ± {stats['reward_std']:.2f}")

rewards = [r["reward_mean"] for r in sweep_results]
print(f"\n{SWEEP_ALGO.upper()} across {len(SWEEP_SEEDS)} seeds:  "
      f"{np.mean(rewards):.2f} ± {np.std(rewards):.2f}")

## Notes

- **Action space:** flat `Discrete(256)` = 4 zones × 4 water levels, decoded inside `env.step`. DQN works without wrappers; PPO/A2C are unaffected.
- **Budget enforcement:** proportional scaling inside `step()` — agents learn scheduling as emergent behaviour.
- **Reward:** per-step water + stress + waterlogging costs, plus a terminal FAO-33 yield bonus. Magnitudes live in `configs/env.yaml`.
- **Weather:** fully synthetic (no network calls), calibrated to Fresno 1991–2020 normals.